In [1]:
import os
from typing import Dict
import time
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)
print(torch.version.cuda)

print(torch.cuda.get_device_name(0))
x = torch.randn(1).to("cuda")
print(x.device)

torch.multiprocessing.set_start_method("spawn", force=True)

# =====================
# CONFIG
# =====================

LOSS_W_ACTION = 0.5
LOSS_W_MOVE = 2.5 # prioritize this
LOSS_W_TERA = 1.0
LOSS_W_SWITCH = 1.5
LOSS_W_FAINT = 0.5

# BATCH_PREFIXES = [
#     r"/home/theha/project/data/tmp_batch0",
#     r"/home/theha/project/data/tmp_batch1",
#     r"/home/theha/project/data/tmp_batch2",
#     r"/home/theha/project/data/tmp_batch3",
#     r"/home/theha/project/data/tmp_batch4",
#     r"/home/theha/project/data/tmp_batch5",
#     r"/home/theha/project/data/tmp_batch6",
#     r"/home/theha/project/data/tmp_batch7",
#     r"/home/theha/project/data/tmp_batch8",
#     r"/home/theha/project/data/tmp_batch9",
# ]

BATCH_PREFIXES = [
    r"/home/theha/project/data/real_batch"
]

# BATCH_PREFIXES = [
#     r"/home/theha/project/data/tmp_batch0",
#     r"/home/theha/project/data/tmp_batch1",
#     r"/home/theha/project/data/tmp_batch2"
# ]

test_y = np.load(r"/home/theha/project/data/real_batch_y_tera.npy", mmap_mode="r")
print("shape:", test_y.shape)
print("first 50 labels:", test_y[:50])
print("uniques + counts:", np.unique(test_y, return_counts=True))

test_y_faint = np.load(r"/home/theha/project/data/real_batch_y_faint.npy", mmap_mode="r")
print("shape:", test_y_faint.shape)
print("first 50 labels:", test_y_faint[:50])
print("uniques + counts:", np.unique(test_y_faint, return_counts=True))

batch_sizes = []
for pref in BATCH_PREFIXES:
    X_path = pref + "_X_hashed.npy"
    n = np.load(X_path, mmap_mode="r").shape[0]
    batch_sizes.append(n)

cum_sizes = np.cumsum([0] + batch_sizes)  # len = num_batches + 1
total_n = cum_sizes[-1]
print("total states:", total_n)

INPUT_DIM = 512        # 512 or 1024 from vectorizer
print(INPUT_DIM)
BATCH_SIZE = 1280
LR = 1e-3
EPOCHS = 100
IGNORE_INDEX = -100            # for CrossEntropyLoss

# Load move / type / species vocabs to get class counts
with open("vocab/shared_moves2id.json", "r") as f:  # TODO: adjust path
    moves2id = json.load(f)
with open("vocab/shared_types2id.json", "r") as f:
    types2id = json.load(f)
with open("vocab/shared_species2id.json", "r") as f:
    species2id = json.load(f)

ACTION_TYPE_MAP_PATH = BATCH_PREFIXES[0] + "_action_type_map.json"  # e.g. "data/gen9ou_full_action_type_map.json"
with open(ACTION_TYPE_MAP_PATH, "r") as f:
    ACTION_TYPE_LABELS = json.load(f)
ACTION_TYPES = list(ACTION_TYPE_LABELS.keys())
N_ACTION_TYPES = len(ACTION_TYPES)
IGNORE_INDEX = -1

N_ACTION_TYPES = len(ACTION_TYPES)               # from your existing ACTION_TYPES list
N_MOVES = len(moves2id)
N_TERA_TYPES = len(types2id)
N_SWITCH_TARGETS = len(species2id)

# Remap -1 from preprocessing to IGNORE_INDEX
def remap_ignore(arr):
    arr = arr.copy()
    arr[arr == -1] = IGNORE_INDEX
    return arr

from dataset import MultiFileTurnsDataset
full_dataset = MultiFileTurnsDataset(BATCH_PREFIXES)
print(full_dataset.perm)

n = len(full_dataset)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)
n_test  = n - n_train - n_val

train_ds, val_ds, test_ds = random_split(full_dataset, [n_train, n_val, n_test])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False,  num_workers=8, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)


cuda
13.0
NVIDIA GeForce RTX 4050 Laptop GPU
cuda:0
shape: (2440277,)
first 50 labels: [-1 -1 -1 -1 -1 -1  9 -1 -1 -1 -1  1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1  4 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1]
uniques + counts: (array([-1,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,
       16, 17], dtype=int32), array([2376121,    1176,    2811,    1701,    1540,    7816,    2667,
          4642,    4754,    5749,    3365,    4015,    1858,    3548,
          2619,     407,     489,    6558,    8441]))
shape: (2440277,)
first 50 labels: [ -1  -1  -1  -1 272 612  -1  -1  -1  -1  -1  -1  -1  -1 284 457  -1  -1
  -1  -1  -1  -1  -1  -1  -1 381  -1  -1  -1 690  -1  -1  -1  -1  -1  -1
 486  -1  -1  -1  -1 690  -1  -1  -1  -1  -1  -1  -1  -1]
uniques + counts: (array([ -1,   0,   1,   2,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  17,  19,  20,  21,  22,  23,  26,  27,  28,  30,  31,
        32,  35,  36,  37,  38,  39,  4

In [ ]:
# =====================
# MODEL
# =====================

class MultiHeadPolicy(nn.Module):
    def __init__(self,
                 input_dim: int,
                 n_action_types: int,
                 n_moves: int,
                 n_tera: int,
                 n_switch_targets: int):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.LayerNorm(1024),
            nn.Dropout(0.1),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.LayerNorm(512),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            nn.ReLU(),
        )
        self.head_action_type = nn.Linear(256, n_action_types)
        self.head_move_id     = nn.Linear(256, n_moves)
        self.head_tera        = nn.Linear(256, n_tera)
        self.head_switch      = nn.Linear(256, n_switch_targets)
        self.head_faint_switch= nn.Linear(256, n_switch_targets)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        h = self.backbone(x)
        return {
            "action_type": self.head_action_type(h),
            "move_id":     self.head_move_id(h),
            "tera":        self.head_tera(h),
            "switch":      self.head_switch(h),
            "faint_switch":self.head_faint_switch(h),
        }


In [3]:
# =====================
# LOSS WITH MASKING
# =====================

def compute_loss(outputs: Dict[str, torch.Tensor],
                 labels: Dict[str, torch.Tensor],
                 ce: nn.CrossEntropyLoss) -> torch.Tensor:
    """
    outputs: dict of (batch, num_classes)
    labels: dict of (batch,) with IGNORE_INDEX where not applicable
    """
    loss_total = 0.0
    weight_total = 0.0

    # action_type: always trained
    t = labels["action_type"]
    logits_t = outputs["action_type"]
    loss_t = ce(logits_t, t)
    loss_total += LOSS_W_ACTION * loss_t
    weight_total += LOSS_W_ACTION

    def masked_head(name: str, weight: float):
        nonlocal loss_total, weight_total
        target = labels[name]
        logits = outputs[name]
        mask = (target != IGNORE_INDEX)
        if mask.any():
            loss = ce(logits[mask], target[mask])
            loss_total += weight * loss
            weight_total += weight

    masked_head("move_id",      LOSS_W_MOVE)
    masked_head("tera",         LOSS_W_TERA)
    masked_head("switch",       LOSS_W_SWITCH)
    masked_head("faint_switch", LOSS_W_FAINT)

    return loss_total / max(weight_total, 1e-8)


In [ ]:
model = MultiHeadPolicy(
    input_dim=INPUT_DIM,
    n_action_types=N_ACTION_TYPES,
    n_moves=N_MOVES,
    n_tera=N_TERA_TYPES,
    n_switch_targets=N_SWITCH_TARGETS,
).to(DEVICE)
print(model)

ce = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
optimizer = optim.Adam(model.parameters(), lr=LR)
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


In [5]:
# =====================
# TRAIN / VAL / TEST SPLIT + TRAINING (NPY)
# =====================
import time

def head_accuracy(logits: torch.Tensor, target: torch.Tensor, ignore_index: int) -> float:
    mask = (target != ignore_index)
    if not mask.any():
        return 0.0
    preds = logits.argmax(dim=1)
    correct = (preds[mask] == target[mask]).sum().item()
    total = mask.sum().item()
    return correct / max(total, 1)

def evaluate(model, loader, ce):
    model.eval()
    total_loss = 0.0
    total_samples = 0

    correct_action_type = 0
    total_action_type   = 0

    correct_move = total_move = 0
    correct_tera = total_tera = 0
    correct_switch = total_switch = 0
    correct_faint = total_faint = 0

    with torch.no_grad():
        for X_batch, Y_batch in loader:
            start = time.perf_counter()
            X_batch = X_batch.to(DEVICE)
            end = time.perf_counter()

            # print(f"forward pass took {end - start:.4f} s")
            Y_batch = {k: v.to(DEVICE) for k, v in Y_batch.items()}

            outputs = model(X_batch)
            loss = compute_loss(outputs, Y_batch, ce)

            bs = X_batch.size(0)
            total_loss += loss.item() * bs
            total_samples += bs

            # action_type
            logits_t = outputs["action_type"]
            target_t = Y_batch["action_type"]
            mask_t = (target_t != IGNORE_INDEX)
            if mask_t.any():
                preds_t = logits_t.argmax(dim=1)
                correct_action_type += (preds_t[mask_t] == target_t[mask_t]).sum().item()
                total_action_type   += mask_t.sum().item()

            # move_id
            logits = outputs["move_id"]
            target = Y_batch["move_id"]
            mask = (target != IGNORE_INDEX)
            if mask.any():
                preds = logits.argmax(dim=1)
                correct_move += (preds[mask] == target[mask]).sum().item()
                total_move   += mask.sum().item()

            # tera
            logits = outputs["tera"]
            target = Y_batch["tera"]
            mask = (target != IGNORE_INDEX)
            if mask.any():
                preds = logits.argmax(dim=1)
                correct_tera += (preds[mask] == target[mask]).sum().item()
                total_tera   += mask.sum().item()

            # switch
            logits = outputs["switch"]
            target = Y_batch["switch"]
            mask = (target != IGNORE_INDEX)
            if mask.any():
                preds = logits.argmax(dim=1)
                correct_switch += (preds[mask] == target[mask]).sum().item()
                total_switch   += mask.sum().item()

            # faint_switch
            logits = outputs["faint_switch"]
            target = Y_batch["faint_switch"]
            mask = (target != IGNORE_INDEX)
            if mask.any():
                preds = logits.argmax(dim=1)
                correct_faint += (preds[mask] == target[mask]).sum().item()
                total_faint   += mask.sum().item()

    avg_loss = total_loss / max(total_samples, 1)
    acc_action = correct_action_type / max(total_action_type, 1) if total_action_type > 0 else 0.0
    acc_move   = correct_move   / max(total_move,   1) if total_move   > 0 else 0.0
    acc_tera   = correct_tera   / max(total_tera,   1) if total_tera   > 0 else 0.0
    acc_switch = correct_switch / max(total_switch, 1) if total_switch > 0 else 0.0
    acc_faint  = correct_faint  / max(total_faint,  1) if total_faint  > 0 else 0.0

    return avg_loss, {
        "action_type": acc_action,
        "move": acc_move,
        "tera": acc_tera,
        "switch": acc_switch,
        "faint": acc_faint,
    }




def train_with_split():
    print("Starting training with train/val/test split...")
    best_val_loss = float("inf")
    best_ckpt_path = None

    for epoch in range(1, EPOCHS + 1):
        print(f"\n===== Epoch {epoch:03d} =====")
        model.train()

        running_loss = 0.0
        total_samples = 0

        # Loop over prefixes one by one (hack)
        for pref in BATCH_PREFIXES:
            print(f"\n--- Training on prefix: {pref} ---")

            # Single-file dataset for this chunk
            train_ds_chunk = MultiFileTurnsDataset([pref])
            train_loader_chunk = DataLoader(
                train_ds_chunk,
                batch_size=BATCH_SIZE,
                shuffle=False,      # rely on dataset's internal perm if used
                num_workers=8,
                pin_memory=True,
            )

            count = 0
            time_total = 0.0
            
            t0 = time.time()
            for X_batch, Y_batch in train_loader_chunk:

                X_batch = X_batch.to(DEVICE)
                Y_batch = {k: v.to(DEVICE) for k, v in Y_batch.items()}

                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = compute_loss(outputs, Y_batch, ce)
                loss.backward()
                optimizer.step()

                bs = X_batch.size(0)
                running_loss += loss.item() * bs
                total_samples += bs
                count += 1

            t1 = time.time()
            if count > 0:
                print(f"Finished {pref}: took {t1-t0:.3f}s over {count} batches")

        # End of epoch: compute train loss over all prefixes
        train_loss = running_loss / max(total_samples, 1)

        # Validation on the fixed val_loader (built once from full_dataset split)
        val_loss, val_accs = evaluate(model, val_loader, ce)

        print(
            f"Epoch {epoch:03d}/{EPOCHS} "
            f"- train_loss: {train_loss:.4f} "
            f"- val_loss: {val_loss:.4f} "
            f"- val_acc(action_type): {val_accs['action_type']:.4f} "
            f"- val_acc(move): {val_accs['move']:.4f} "
            f"- val_acc(tera): {val_accs['tera']:.4f} "
            f"- val_acc(switch): {val_accs['switch']:.4f} "
            f"- val_acc(faint): {val_accs['faint']:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_ckpt_path = os.path.join(CHECKPOINT_DIR, "policy_best.pt")
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
            }, best_ckpt_path)

    print(f"Best val_loss: {best_val_loss:.4f}, checkpoint: {best_ckpt_path}")

    if best_ckpt_path is not None:
        ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state_dict"])

    test_loss, test_acc = evaluate(model, test_loader, ce)
    print(f"TEST - loss: {test_loss:.4f}, action_type acc: {test_acc}")

    return model

# Run training
# trained_model = train_with_split()
best_ckpt_path = os.path.join(CHECKPOINT_DIR, "policy_best.pt")
if best_ckpt_path is not None:
    ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])

test_loss, test_acc = evaluate(model, test_loader, ce)
print(f"TEST - loss: {test_loss:.4f}, action_type acc: {test_acc:}")



TEST - loss: 2.2603, action_type acc: {'action_type': 0.6461813964733699, 'move': 0.2890628344289864, 'tera': 0.5241797543150365, 'switch': 0.3173594579747299, 'faint': 0.23916599839615077}


<All keys matched successfully>

In [ ]:
'''Testing Batch size 3k with it'''

BATCH_SIZE = 3000

X_np = np.load(BATCH_PREFIXES[0] + "_X_hashed.npy", mmap_mode="r")
N = X_np.shape[0]
print("Total states:", N)

# take first 1000 (or fewer if N < 1000)
n_batch = min(BATCH_SIZE, N)
X_batch_np = X_np[:n_batch].astype("float32")
X_batch = torch.from_numpy(X_batch_np).to(DEVICE)

N_ACTION_TYPES = len(ACTION_TYPE_LABELS)
N_MOVES = len(moves2id)
N_TERA = len(types2id)
N_SWITCH = len(species2id)

model = MultiHeadPolicy(
    input_dim=INPUT_DIM,
    n_action_types=N_ACTION_TYPES,
    n_moves=N_MOVES,
    n_tera=N_TERA_TYPES,
    n_switch_targets=N_SWITCH_TARGETS,
).to(DEVICE)
print(model)
best_ckpt_path = os.path.join(CHECKPOINT_DIR, "policy_best.pt")
ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])


start = time.time()
with torch.no_grad():
    outputs = model(X_batch)
end = time.time()
for k, v in outputs.items():
    print(k, v.shape)

In [7]:
for pref in BATCH_PREFIXES:
    y = np.load(pref + "_y_winner.npy", mmap_mode="r")
    print(pref, np.unique(y, return_counts=True))

from dataset import MultiFileWinDataset
full_win_ds = MultiFileWinDataset(BATCH_PREFIXES)

n = len(full_win_ds)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)
n_test  = n - n_train - n_val
win_train_ds, win_val_ds, win_test_ds = random_split(full_win_ds, [n_train, n_val, n_test])

win_train_loader = DataLoader(win_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=8, pin_memory=True)
win_val_loader   = DataLoader(win_val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=8, pin_memory=True)
win_test_loader  = DataLoader(win_test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=8, pin_memory=True)


/home/theha/project/data/real_batch (array([-1,  0,  1], dtype=int32), array([   1905, 1231505, 1206867]))


In [9]:
class WinPredictor(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
        )
        self.head = nn.Linear(256, 1)  # logit for P1 win

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.backbone(x)
        logit = self.head(h).squeeze(-1)  # (batch,)
        return logit

win_model = WinPredictor(INPUT_DIM).to(DEVICE)
bce = nn.BCEWithLogitsLoss()
win_optimizer = optim.Adam(win_model.parameters(), lr=LR)
BEST_WIN_CKPT = "win_best.pt"


In [ ]:
def eval_win(model, loader):
    model.eval()
    total_loss = 0.0
    total_samples = 0
    correct = 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            logits = model(X_batch)
            loss = bce(logits, y_batch)

            bs = X_batch.size(0)
            total_loss += loss.item() * bs
            total_samples += bs

            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()
            correct += (preds == y_batch).sum().item()

    avg_loss = total_loss / max(total_samples, 1)
    acc = correct / max(total_samples, 1)
    return avg_loss, acc

def train_win_with_split():
    print("Starting win predictor training...")
    best_val_loss = float("inf")
    best_ckpt_path = None

    for epoch in range(1, EPOCHS + 1):
        win_model.train()
        running_loss = 0.0
        total_samples = 0

        for X_batch, y_batch in win_train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            win_optimizer.zero_grad()
            logits = win_model(X_batch)
            loss = bce(logits, y_batch)
            loss.backward()
            win_optimizer.step()

            bs = X_batch.size(0)
            running_loss += loss.item() * bs
            total_samples += bs

        train_loss = running_loss / max(total_samples, 1)
        val_loss, val_acc = eval_win(win_model, win_val_loader)

        print(
            f"Epoch {epoch:03d}/{EPOCHS} "
            f"- train_loss: {train_loss:.4f} "
            f"- val_loss: {val_loss:.4f} "
            f"- val_acc(win): {val_acc:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_ckpt_path = BEST_WIN_CKPT
            torch.save({
                "epoch": epoch,
                "model_state_dict": win_model.state_dict(),
                "optimizer_state_dict": win_optimizer.state_dict(),
                "val_loss": val_loss,
            }, best_ckpt_path)

    print(f"Best val_loss: {best_val_loss:.4f}, checkpoint: {best_ckpt_path}")

    if best_ckpt_path is not None:
        ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
        win_model.load_state_dict(ckpt["model_state_dict"])

    test_loss, test_acc = eval_win(win_model, win_test_loader)
    print(f"WIN TEST - loss: {test_loss:.4f}, acc: {test_acc:.4f}")

    return win_model

# Train:
win_model = train_win_with_split()

# Later, to load best:
best_win_ckpt_path = BEST_WIN_CKPT
ckpt = torch.load(best_win_ckpt_path, map_location=DEVICE)
win_model = WinPredictor(INPUT_DIM).to(DEVICE)
win_model.load_state_dict(ckpt["model_state_dict"])
win_model.eval()


Starting win predictor training...
Epoch 001/100 - train_loss: 0.7152 - val_loss: 0.6474 - val_acc(win): 0.5966
Epoch 002/100 - train_loss: 0.6456 - val_loss: 0.6303 - val_acc(win): 0.6253
Epoch 003/100 - train_loss: 0.6283 - val_loss: 0.6172 - val_acc(win): 0.6411
Epoch 004/100 - train_loss: 0.6150 - val_loss: 0.6072 - val_acc(win): 0.6498
Epoch 005/100 - train_loss: 0.6117 - val_loss: 0.6255 - val_acc(win): 0.6235
Epoch 006/100 - train_loss: 0.6040 - val_loss: 0.6121 - val_acc(win): 0.6328
Epoch 007/100 - train_loss: 0.5953 - val_loss: 0.5905 - val_acc(win): 0.6603
Epoch 008/100 - train_loss: 0.5865 - val_loss: 0.5816 - val_acc(win): 0.6711
Epoch 009/100 - train_loss: 0.5812 - val_loss: 0.5818 - val_acc(win): 0.6720
Epoch 010/100 - train_loss: 0.5733 - val_loss: 0.5958 - val_acc(win): 0.6508
Epoch 011/100 - train_loss: 0.5655 - val_loss: 0.5651 - val_acc(win): 0.6828
Epoch 012/100 - train_loss: 0.5603 - val_loss: 0.5605 - val_acc(win): 0.6874
Epoch 013/100 - train_loss: 0.5547 - val_

In [ ]:
def p1_win_prob(x_np: np.ndarray) -> float:
    win_model.eval()
    with torch.no_grad():
        x = torch.from_numpy(x_np).float().unsqueeze(0).to(DEVICE)
        logit = win_model(x).item()
        prob = torch.sigmoid(torch.tensor(logit)).item()
    return prob  # between 0 and 1
